In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter
import fsspec

# ── Load Data ──────────────────────────────────────────────────────────────────
def load_specific_parquet(url):
    with fsspec.open(url, 'rb') as f:
        return pd.read_parquet(f)

url  = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro.parquet"
metro_24_25 = load_specific_parquet(url)

#---------------Loading the corridor list and cleaning it -------------------------
flight_route = pd.read_excel("Flight Paris for Size the Prize v1.xlsx")
corridors = flight_route[~flight_route['Row Labels'].str.contains('Unknown', na=False)]['Row Labels'].str.strip().iloc[:-1]
corridors = corridors.str.replace(' - ','->')

# Filter out corridors where From and To are identical
corridors = corridors[corridors.apply(lambda x: x.split('->')[0] != x.split('->')[1])]

# copying the Data
Data = metro_24_25.copy()


In [24]:
LIGHT_JET_MODELS = [
    'Phenom 300', 'Phenom 300-E', 'Citation CJ3', 'Citation CJ3+',
    'Citation CJ4', 'Hawker 400XP', 'Citation CJ2+', 'Citation Ultra',
    'Citation V', '400-A', 'Citation CJ4 Gen2', 'Citation Encore+'
]
smid = Data[Data['aircraft_segment']=='Super Midsize Jet']
smid[smid['aircraft_model'].isin(LIGHT_JET_MODELS)]

,FlightDate_utc,Operator,FromAirport,FromState,ToAirport,ToState,Hours,aircraft_tailsign,aircraft_model,aircraft_segment,fuel_uplift_usg,FlightDate_ET,year,month,dow,hour,FromMetro,ToMetro


In [25]:
import pandas as pd
import numpy as np
# ── Precompute lookup arrays outside the loop ──────────────────────────────────────────
Thresh = False
DAY_ORDER    = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
BUCKET_ORDER = ['07:00–10:00', '10:00–13:00', '13:00–16:00',
                '16:00–19:00', '19:00–24:00', '00:00–07:00']
# Vectorized hour → bucket via numpy array indexing (hours 0–23)
_HOUR_TO_BUCKET = np.array(
    ['00:00–07:00'] * 7 +   # hours 0–6
    ['07:00–10:00'] * 3 +   # hours 7–9
    ['10:00–13:00'] * 3 +   # hours 10–12
    ['13:00–16:00'] * 3 +   # hours 13–15
    ['16:00–19:00'] * 3 +   # hours 16–18
    ['19:00–24:00'] * 5     # hours 19–23
)
# Vectorized pct → tier via pd.cut (replaces map + lambda)
TIER_BINS   = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, float('inf')]
TIER_LABELS = ['0 - 1','1 - 2','2 - 3','3 - 4','4 - 5',
               '5 - 6','6 - 7','7 - 8','8 - 9','9 - 10','10+']
# Light Jet specific aircraft models
LIGHT_JET_MODELS = [
    'Phenom 300', 'Phenom 300-E', 'Citation CJ3', 'Citation CJ3+',
    'Citation CJ4', 'Hawker 400XP', 'Citation CJ2+', 'Citation Ultra',
    'Citation V', '400-A', 'Citation CJ4 Gen2', 'Citation Encore+'
]
# ── Preprocess ONCE before the loop ───────────────────────────────────────────────────
data = Data.iloc[:, 1:].copy()
# filter all the corridors that have same FromMetro and ToMetro
data = data[data['FromMetro'] != data['ToMetro']]
# filtering the Aircraft model (Light Jet specific models)
data = data[data['aircraft_model'].isin(LIGHT_JET_MODELS)]
# Build route column
data['Route'] = data['FromMetro'] + '->' + data['ToMetro']
# Parse datetime once
data['dep_dt']   = pd.to_datetime(data['FlightDate_ET'], errors='coerce')
data['dep_hour'] = data['dep_dt'].dt.hour
# Day name as ordered categorical (created once)
data['day_name'] = pd.Categorical(
    data['dep_dt'].dt.day_name(),
    categories=DAY_ORDER,
    ordered=True
)
# Hour bucket via vectorized numpy indexing (created once)
data['hour_bucket'] = pd.Categorical(
    _HOUR_TO_BUCKET[data['dep_hour'].to_numpy()],
    categories=BUCKET_ORDER,
    ordered=True
)
valid_corridors = list(set(corridors).intersection(data['Route'].unique()))
# ── Filter to only required corridors ─────────────────────────────────────────────────
missing = [cor for cor in corridors if cor not in data['Route'].values]
if missing:
    print(f"Note: The following corridors were not found in data and will be skipped: {missing}")
data_filtered = data[data['Route'].isin(corridors)]
# ── Single groupby across ALL corridors (no loop needed) ──────────────────────────────
final_report = (
    data_filtered
    .groupby(['Route', 'day_name', 'hour_bucket'], observed=False)
    .size()
    .reset_index(name='flight_count')
)
# Total flights per Route + Day
final_report['total_flights'] = (
    final_report
    .groupby('Route')['flight_count']
    .transform('sum')
)
# Percentage
final_report['pct_flights'] = (
    (final_report['flight_count'] / final_report['total_flights']) * 100
)
# Vectorized tier assignment via pd.cut
final_report['Tier_gap'] = pd.cut(
    final_report['pct_flights'],
    bins=TIER_BINS,
    labels=TIER_LABELS,
    right=False
)
# 1. Convert Tier_gap to string to allow the new 'insufficient' label
final_report['Tier_gap'] = final_report['Tier_gap'].astype(str)
# 2. Update Tier_gap where flight_count is less than threshold
if Thresh:
    final_report.loc[final_report['flight_count'] < Thresh, 'Tier_gap'] = 'insufficient'

Note: The following corridors were not found in data and will be skipped: ['New York City->Bay Area', 'DMV->Bay Area', 'New York City->LA Basin', 'DMV->LA Basin', 'North Florida->Bay Area', 'South Florida->Bay Area', 'Atlanta->Bay Area', 'North Florida->LA Basin', 'Boston->Phoenix Valley', 'Bay Area->Philadelphia', 'Philadelphia->LA Basin', 'South Florida->Seattle', 'Boston->Bay Area', 'New York City->Seattle', 'Philadelphia->Bay Area', 'Boston->Seattle', 'Philadelphia->Phoenix Valley', 'North Florida->Seattle']


In [27]:
final_report.head()

,Route,day_name,hour_bucket,flight_count,total_flights,pct_flights,Tier_gap
0,Atlanta->Boston,Monday,07:00–10:00,10,504,1.984127,1 - 2
1,Atlanta->Boston,Monday,10:00–13:00,24,504,4.761905,4 - 5
2,Atlanta->Boston,Monday,13:00–16:00,15,504,2.976190,2 - 3
3,Atlanta->Boston,Monday,16:00–19:00,5,504,0.992063,0 - 1
4,Atlanta->Boston,Monday,19:00–24:00,2,504,0.396825,0 - 1


In [28]:
DI_LI = pd.read_excel("DI_Analysis_LJ_SMID.xlsx",sheet_name=None)

In [30]:
smid = DI_LI['DI SMID']

In [32]:
smid.head()

,Route,day_name,hour_bucket,flight_count,total_flights,pct_flights,Tier_gap
0,Atlanta->Bay Area,Monday,07:00–10:00,17,429,3.962704,3 - 4
1,Atlanta->Bay Area,Monday,10:00–13:00,23,429,5.361305,5 - 6
2,Atlanta->Bay Area,Monday,13:00–16:00,13,429,3.030303,3 - 4
3,Atlanta->Bay Area,Monday,16:00–19:00,12,429,2.797203,2 - 3
4,Atlanta->Bay Area,Monday,19:00–24:00,4,429,0.932401,0 - 1


In [33]:
# Define the filename
output_file = 'DI_Analysis_LJ_SMID.xlsx'

# Use ExcelWriter to save multiple sheets
with pd.ExcelWriter(output_file) as writer:
    final_report.to_excel(writer, sheet_name='DI Light Jet', index=False)
    smid.to_excel(writer, sheet_name='DI SMID', index=False)

print(f"Successfully saved to {output_file}")

Successfully saved to DI_Analysis_LJ_SMID.xlsx
